**Machine Learning Classification models and Evaluation metrics**

In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# Step 1: Preprocessing and Justification
print("Justification: We must scale features using StandardScaler because distance-based algorithms like KNN and SVM perform poorly if features have different units.")

# Load the dataset
df = pd.read_csv('spotify_songs.csv')

# Drop string columns, keep only the 13 numeric features
features = ['track_popularity', 'danceability', 'energy', 'key', 'loudness',
            'mode', 'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo', 'duration_ms']
X = df[features]
y = df['playlist_genre']

# Convert text target to numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Creating model folder for deployment requirements
os.makedirs('model', exist_ok=True)
joblib.dump(encoder, 'model/label_encoder.pkl')

# Train/Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Save test dataset for Streamlit UI upload
test_data_to_save = X_test.copy()
test_data_to_save['playlist_genre'] = encoder.inverse_transform(y_test)
test_data_to_save.to_csv('test_data.csv', index=False)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'model/scaler.pkl')

# Step 2: Implementation of 5 Machine Learning Models and Evaluation Metrics
print("\nFormulas being calculated for all models:")
print("- Accuracy: (TP + TN) / (Total)")
print("- Precision: TP / (TP + FP)")
print("- Recall: TP / (TP + FN)")
print("- F1 Score: 2 * (Precision * Recall) / (Precision + Recall)")
print("- MCC: (TP*TN - FP*FN) / sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))")
print("- AUC: Area under the ROC Curve (calculated via One-vs-Rest for multiclass)\n")

# Defining the 5 required models
models_dict = {
    'Logistic Regression': LogisticRegression(max_iter=2000, multi_class='multinomial'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

model_results = []

for model_name, clf in models_dict.items():
    print(f"Training {model_name}...")

    # Training step
    clf.fit(X_train_scaled, y_train)

    # Prediction steps
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)

    # Calculating all required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    model_results.append({
        'ML Model Name': model_name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

    # Save each model for the deployment step
    filename = f"model/{model_name.replace(' ', '_').lower()}.pkl"
    joblib.dump(clf, filename)

print("\n--- Final Model Comparison Table ---")
df_results = pd.DataFrame(model_results)
print(df_results.to_markdown(index=False))

Justification: We must scale features using StandardScaler because distance-based algorithms like KNN and SVM perform poorly if features have different units.

Formulas being calculated for all models:
- Accuracy: (TP + TN) / (Total)
- Precision: TP / (TP + FP)
- Recall: TP / (TP + FN)
- F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
- MCC: (TP*TN - FP*FN) / sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
- AUC: Area under the ROC Curve (calculated via One-vs-Rest for multiclass)

Training Logistic Regression...


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training Decision Tree...
Training KNN...
Training Naive Bayes...
Training Support Vector Machine...

--- Final Model Comparison Table ---
| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |     0.4713 | 0.7882 |      0.4609 |   0.4706 | 0.4634 | 0.3656 |
| Decision Tree          |     0.4421 | 0.6749 |      0.4423 |   0.4418 | 0.442  | 0.3299 |
| KNN                    |     0.4515 | 0.756  |      0.457  |   0.4494 | 0.4499 | 0.3417 |
| Naive Bayes            |     0.4515 | 0.7711 |      0.455  |   0.4494 | 0.4493 | 0.3419 |
| Support Vector Machine |     0.5369 | 0.838  |      0.5327 |   0.5357 | 0.5321 | 0.4442 |
